In [2]:
!pip install -q transformers datasets accelerate

In [3]:


from __future__ import annotations

import argparse
import ast
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


DEFAULT_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DEFAULT_SYSTEM_PROMPT = "You are an expert Python competitive programmer."


@dataclass
class Task:
    task_id: str
    prompt: str
    test: str
    entry_point: str


@dataclass
class TokenTrace:
    position: int
    token_id: int
    token_text: str
    entropy: float
    margin: float
    top1_id: int
    top1_text: str
    top2_id: int
    top2_text: str


@dataclass
class Generation:
    token_ids: list[int]
    code: str
    trace: list[TokenTrace]
    latency_s: float


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--model-name", default=DEFAULT_MODEL)
    parser.add_argument("--output-dir", default="/kaggle/working/top2_counterfactual_pilot")
    parser.add_argument("--task-ids", default="", help="Comma-separated HumanEval task IDs.")
    parser.add_argument("--num-tasks", type=int, default=10, help="Used only when --task-ids is empty.")
    parser.add_argument("--max-new-tokens", type=int, default=256)
    parser.add_argument("--candidates-per-task", type=int, default=3)
    parser.add_argument("--controls-per-task", type=int, default=3)
    parser.add_argument("--edge-buffer", type=int, default=5)
    parser.add_argument("--timeout-s", type=int, default=8)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--trust-remote-code", action="store_true")
    parser.add_argument("--overwrite", action="store_true")
    # Jupyter/Kaggle executes a cell with an internal ``-f <kernel.json>``
    # argument. Accept that one argument pair while keeping normal CLI typos
    # visible to the user.
    args, unknown = parser.parse_known_args()
    if unknown:
        if len(unknown) == 2 and unknown[0] == "-f":
            return args
        parser.error(f"unrecognized arguments: {' '.join(unknown)}")
    return args


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def load_tasks(task_ids: str, num_tasks: int) -> list[Task]:
    dataset = load_dataset("openai_humaneval", split="test")
    requested_ids = {item.strip() for item in task_ids.split(",") if item.strip()}
    tasks = [
        Task(
            task_id=row["task_id"],
            prompt=row["prompt"],
            test=row["test"],
            entry_point=row["entry_point"],
        )
        for row in dataset
        if not requested_ids or row["task_id"] in requested_ids
    ]
    if requested_ids:
        missing = requested_ids - {task.task_id for task in tasks}
        if missing:
            raise ValueError(f"Unknown HumanEval task IDs: {sorted(missing)}")
        return tasks
    return tasks[:num_tasks]


def build_prompt(task: Task, tokenizer: Any) -> str:
    user_prompt = (
        "Complete the following Python function.\n"
        "Return only valid Python code. Do not use Markdown. Do not explain.\n\n"
        f"{task.prompt}"
    )
    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f"{DEFAULT_SYSTEM_PROMPT}\n\n{user_prompt}"


def strip_markdown_fences(text: str) -> str:
    text = (text or "").strip()
    blocks = re.findall(r"```(?:python|py)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    blocks = [block.strip() for block in blocks if block.strip()]
    if blocks:
        keywords = ("def ", "import ", "from ", "class ", "return ", "assert ")
        return max(blocks, key=lambda block: sum(key in block for key in keywords) * 10 + len(block))
    return text.replace("```python", "").replace("```py", "").replace("```", "").strip()


def extract_code(raw_output: str, entry_point: str) -> str:
    text = strip_markdown_fences(raw_output)
    for marker in ("Explanation:", "Example:", "Examples:", "# Explanation"):
        marker_index = text.find(marker)
        if marker_index != -1:
            text = text[:marker_index].strip()
    match = re.search(rf"def\s+{re.escape(entry_point)}\s*\(", text)
    if match:
        imports = [
            line.strip()
            for line in text[: match.start()].splitlines()
            if line.strip().startswith(("import ", "from "))
        ]
        function_code = text[match.start() :].strip()
        return "\n".join(imports + ([""] if imports else []) + [function_code]).strip()
    return text.strip()


def evaluate(task: Task, raw_output: str, timeout_s: int) -> tuple[bool, str | None]:
    code = extract_code(raw_output, task.entry_point)
    try:
        ast.parse(code)
    except SyntaxError as error:
        return False, f"SyntaxError: {error.msg} at line {error.lineno}"

    prelude = (
        "from typing import *\nimport math\nimport re\nimport itertools\n"
        "import collections\nimport functools\nimport heapq\nimport bisect\n"
        "import string\nimport statistics\nfrom collections import *\n\n"
    )
    source = prelude + code + "\n\n" + task.test + f"\n\ncheck({task.entry_point})\n"
    with tempfile.TemporaryDirectory() as temp_dir:
        candidate_path = Path(temp_dir) / "candidate.py"
        candidate_path.write_text(source, encoding="utf-8")
        try:
            result = subprocess.run(
                [sys.executable, str(candidate_path)],
                cwd=temp_dir,
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )
        except subprocess.TimeoutExpired:
            return False, f"Timeout: exceeded {timeout_s}s"
    if result.returncode == 0:
        return True, None
    stderr = (result.stderr or result.stdout or "unknown execution failure").strip()
    return False, stderr[-800:]


def model_input_device(model: Any) -> torch.device:
    return model.get_input_embeddings().weight.device


def entropy_and_top2(logits: torch.Tensor, tokenizer: Any, position: int) -> TokenTrace:
    logits = logits.float()
    log_probabilities = torch.log_softmax(logits, dim=-1)
    probabilities = log_probabilities.exp()
    entropy = float((-(probabilities * log_probabilities).sum()).item())
    top_values, top_ids = torch.topk(logits, k=2, dim=-1)
    top1_id, top2_id = int(top_ids[0].item()), int(top_ids[1].item())
    return TokenTrace(
        position=position,
        token_id=top1_id,
        token_text=tokenizer.decode([top1_id]),
        entropy=entropy,
        margin=float((top_values[0] - top_values[1]).item()),
        top1_id=top1_id,
        top1_text=tokenizer.decode([top1_id]),
        top2_id=top2_id,
        top2_text=tokenizer.decode([top2_id]),
    )


@torch.inference_mode()
def greedy_generate(model: Any, tokenizer: Any, prompt: str, max_new_tokens: int) -> Generation:
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    started = time.perf_counter()
    outputs = model(prompt_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    generated_ids: list[int] = []
    trace: list[TokenTrace] = []
    eos_token_id = tokenizer.eos_token_id

    for position in range(max_new_tokens):
        token_trace = entropy_and_top2(next_logits[0], tokenizer, position)
        trace.append(token_trace)
        next_token_id = token_trace.top1_id
        generated_ids.append(next_token_id)
        if eos_token_id is not None and next_token_id == eos_token_id:
            break
        next_token = torch.tensor([[next_token_id]], device=input_device, dtype=torch.long)
        outputs = model(next_token, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    return Generation(
        token_ids=generated_ids,
        code=tokenizer.decode(generated_ids, skip_special_tokens=True),
        trace=trace,
        latency_s=time.perf_counter() - started,
    )


@torch.inference_mode()
def generate_top1_top2_batch(
    model: Any,
    tokenizer: Any,
    prompt: str,
    prefix_ids: list[int],
    top1_id: int,
    top2_id: int,
    max_new_tokens: int,
) -> tuple[Generation, Generation]:
    """Continue top-1 and top-2 branches in a batch of two equally long prefixes."""

    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=input_device).unsqueeze(0)
    shared_prefix = torch.cat([prompt_ids, prefix_tensor], dim=1) if prefix_ids else prompt_ids
    candidates = torch.tensor([[top1_id], [top2_id]], dtype=torch.long, device=input_device)
    input_ids = torch.cat([shared_prefix.repeat(2, 1), candidates], dim=1)
    started = time.perf_counter()
    outputs = model(input_ids, use_cache=True)
    cache = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :]
    branch_ids = [[top1_id], [top2_id]]
    branch_trace: list[list[TokenTrace]] = [[], []]
    finished = [False, False]
    eos_token_id = tokenizer.eos_token_id
    remaining = max(max_new_tokens - len(prefix_ids) - 1, 0)

    for step in range(remaining):
        next_ids: list[int] = []
        for branch_index in range(2):
            token_trace = entropy_and_top2(next_logits[branch_index], tokenizer, len(prefix_ids) + 1 + step)
            branch_trace[branch_index].append(token_trace)
            token_id = eos_token_id if finished[branch_index] and eos_token_id is not None else token_trace.top1_id
            branch_ids[branch_index].append(int(token_id))
            if eos_token_id is not None and token_id == eos_token_id:
                finished[branch_index] = True
            next_ids.append(int(token_id))
        if all(finished):
            break
        next_tensor = torch.tensor(next_ids, dtype=torch.long, device=input_device).unsqueeze(1)
        outputs = model(next_tensor, past_key_values=cache, use_cache=True)
        cache = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :]

    latency_s = time.perf_counter() - started
    complete_ids = [prefix_ids + branch for branch in branch_ids]
    return (
        Generation(
            token_ids=complete_ids[0],
            code=tokenizer.decode(complete_ids[0], skip_special_tokens=True),
            trace=branch_trace[0],
            latency_s=latency_s,
        ),
        Generation(
            token_ids=complete_ids[1],
            code=tokenizer.decode(complete_ids[1], skip_special_tokens=True),
            trace=branch_trace[1],
            latency_s=latency_s,
        ),
    )


def select_positions(
    trace: list[TokenTrace],
    candidates_per_task: int,
    controls_per_task: int,
    edge_buffer: int,
    rng: np.random.Generator,
) -> list[tuple[str, TokenTrace]]:
    valid = trace[edge_buffer : max(len(trace) - edge_buffer, edge_buffer)]
    high_entropy = sorted(valid, key=lambda item: item.entropy, reverse=True)[:candidates_per_task]
    high_positions = {item.position for item in high_entropy}
    controls_pool = [item for item in valid if item.position not in high_positions]
    controls_count = min(controls_per_task, len(controls_pool))
    controls = (
        [controls_pool[index] for index in rng.choice(len(controls_pool), size=controls_count, replace=False)]
        if controls_count
        else []
    )
    return [("high_entropy", item) for item in high_entropy] + [("random_control", item) for item in controls]


def baseline_record(task: Task, generation: Generation, passed: bool, error: str | None) -> dict[str, Any]:
    return {
        "task_id": task.task_id,
        "passed": passed,
        "error": error,
        "code": generation.code,
        "token_ids": generation.token_ids,
        "trace": [asdict(item) for item in generation.trace],
        "generation_latency_s": generation.latency_s,
    }


def write_summary(branch_records: list[dict[str, Any]], output_dir: Path) -> None:
    rows = []
    for selection_type in ("high_entropy", "random_control"):
        group = [record for record in branch_records if record["selection_type"] == selection_type]
        if not group:
            continue
        recoverable = sum(bool(record["recoverable"]) for record in group)
        rows.append(
            {
                "selection_type": selection_type,
                "n_positions": len(group),
                "recoverable": recoverable,
                "recovery_rate": recoverable / len(group),
                "mean_entropy": float(np.mean([record["entropy"] for record in group])),
                "mean_extra_tokens": float(np.mean([record["extra_tokens"] for record in group])),
            }
        )
    report = {"rows": rows, "created_at_unix": time.time()}
    (output_dir / "summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    markdown = ["# Top-1 vs Top-2 counterfactual pilot", "", "| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |", "|---|---:|---:|---:|---:|---:|"]
    markdown.extend(
        "| {selection_type} | {n_positions} | {recoverable} | {recovery_rate:.1%} | {mean_entropy:.3f} | {mean_extra_tokens:.1f} |".format(**row)
        for row in rows
    )
    (output_dir / "summary.md").write_text("\n".join(markdown) + "\n", encoding="utf-8")
    print("\n".join(markdown))


def main() -> None:
    args = parse_args()
def run_notebook(
    *,
    task_ids: str,
    model_name: str = DEFAULT_MODEL,
    output_dir: str = "/kaggle/working/top2_counterfactual_pilot",
    max_new_tokens: int = 256,
    candidates_per_task: int = 3,
    controls_per_task: int = 3,
    edge_buffer: int = 5,
    timeout_s: int = 8,
    seed: int = 42,
    trust_remote_code: bool = False,
    overwrite: bool = False,
) -> None:
    """Run the pilot directly from a Kaggle notebook cell.

    Example:
        run_notebook(task_ids="HumanEval/26,HumanEval/38", candidates_per_task=2)
    """

    args = argparse.Namespace(
        model_name=model_name,
        output_dir=output_dir,
        task_ids=task_ids,
        num_tasks=10,
        max_new_tokens=max_new_tokens,
        candidates_per_task=candidates_per_task,
        controls_per_task=controls_per_task,
        edge_buffer=edge_buffer,
        timeout_s=timeout_s,
        seed=seed,
        trust_remote_code=trust_remote_code,
        overwrite=overwrite,
    )
    main(args)


def main(args: argparse.Namespace | None = None) -> None:
    args = args or parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("This pilot requires a Kaggle GPU session.")
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    rng = np.random.default_rng(args.seed)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    baseline_path = output_dir / "baselines.jsonl"
    branch_path = output_dir / "branches.jsonl"
    metadata_path = output_dir / "metadata.json"
    if args.overwrite:
        for path in (baseline_path, branch_path, metadata_path):
            if path.exists():
                path.unlink()

    tasks = load_tasks(args.task_ids, args.num_tasks)
    metadata_path.write_text(
        json.dumps({"args": vars(args), "tasks": [task.task_id for task in tasks]}, indent=2),
        encoding="utf-8",
    )
    print(f"Loading one model across {torch.cuda.device_count()} GPU(s): {args.model_name}")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=args.trust_remote_code)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=args.trust_remote_code,
    )
    model.eval()

    existing_baselines = {record["task_id"]: record for record in read_jsonl(baseline_path)}
    existing_branches = {record["task_id"] for record in read_jsonl(branch_path)}
    for task in tasks:
        if task.task_id not in existing_baselines:
            print(f"Baseline {task.task_id}")
            prompt = build_prompt(task, tokenizer)
            generation = greedy_generate(model, tokenizer, prompt, args.max_new_tokens)
            passed, error = evaluate(task, generation.code, args.timeout_s)
            record = baseline_record(task, generation, passed, error)
            append_jsonl(baseline_path, record)
            existing_baselines[task.task_id] = record
            print(f"  {'PASS' if passed else 'FAIL'} | {len(generation.token_ids)} tokens")

        baseline = existing_baselines[task.task_id]
        if baseline["passed"]:
            print(f"Skip {task.task_id}: baseline passed (pilot targets failures).")
            continue
        if task.task_id in existing_branches:
            print(f"Skip {task.task_id}: branches already saved.")
            continue

        prompt = build_prompt(task, tokenizer)
        trace = [TokenTrace(**item) for item in baseline["trace"]]
        selected = select_positions(
            trace,
            candidates_per_task=args.candidates_per_task,
            controls_per_task=args.controls_per_task,
            edge_buffer=args.edge_buffer,
            rng=rng,
        )
        if not selected:
            print(f"Skip {task.task_id}: too few generated tokens for valid branch positions.")
            continue
        print(f"Branching {task.task_id}: {len(selected)} positions")
        for selection_type, point in selected:
            prefix_ids = baseline["token_ids"][: point.position]
            top1_generation, top2_generation = generate_top1_top2_batch(
                model,
                tokenizer,
                prompt,
                prefix_ids=prefix_ids,
                top1_id=point.top1_id,
                top2_id=point.top2_id,
                max_new_tokens=args.max_new_tokens,
            )
            top1_passed, top1_error = evaluate(task, top1_generation.code, args.timeout_s)
            top2_passed, top2_error = evaluate(task, top2_generation.code, args.timeout_s)
            record = {
                "task_id": task.task_id,
                "selection_type": selection_type,
                "position": point.position,
                "position_relative": point.position / max(len(baseline["token_ids"]) - 1, 1),
                "entropy": point.entropy,
                "margin": point.margin,
                "top1_token": point.top1_text,
                "top2_token": point.top2_text,
                "top1_passed": top1_passed,
                "top1_error": top1_error,
                "top2_passed": top2_passed,
                "top2_error": top2_error,
                "recoverable": (not top1_passed) and top2_passed,
                "top1_matches_baseline": top1_generation.code == baseline["code"],
                "extra_tokens": len(top1_generation.token_ids) + len(top2_generation.token_ids) - 2 * len(prefix_ids),
                "branch_latency_s": top1_generation.latency_s,
                "top1_code": top1_generation.code,
                "top2_code": top2_generation.code,
            }
            append_jsonl(branch_path, record)
            print(
                f"  {selection_type} t={point.position:>3} H={point.entropy:.3f} "
                f"top1={'P' if top1_passed else 'F'} top2={'P' if top2_passed else 'F'}"
            )
        existing_branches.add(task.task_id)
        write_summary(read_jsonl(branch_path), output_dir)

    write_summary(read_jsonl(branch_path), output_dir)
    print(f"\nSaved resumable outputs to: {output_dir}")


if __name__ == "__main__":
    main()

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loading one model across 2 GPU(s): Qwen/Qwen2.5-Coder-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Skip HumanEval/0: baseline passed (pilot targets failures).
Skip HumanEval/1: baseline passed (pilot targets failures).
Skip HumanEval/2: baseline passed (pilot targets failures).
Skip HumanEval/3: baseline passed (pilot targets failures).
Skip HumanEval/4: baseline passed (pilot targets failures).
Skip HumanEval/5: baseline passed (pilot targets failures).
Skip HumanEval/6: baseline passed (pilot targets failures).
Skip HumanEval/7: baseline passed (pilot targets failures).
Skip HumanEval/8: baseline passed (pilot targets failures).
Skip HumanEval/9: baseline passed (pilot targets failures).
# Top-1 vs Top-2 counterfactual pilot

| Selection | Positions | Recovered | Recovery rate | Mean entropy | Extra tokens |
|---|---:|---:|---:|---:|---:|
| high_entropy | 1 | 0 | 0.0% | 0.891 | 192.0 |
| random_control | 1 | 0 | 0.0% | 0.000 | 90.0 |

Saved resumable outputs to: /kaggle/working/top2_counterfactual_pilot


In [3]:
"""Pilot: entropy vs. true top-1/top-2 probability margin on HumanEval/26.

Paste this file into a NEW Kaggle cell after running:
  1. the dependency-installation cell; and
  2. the large definitions cell from notebook0cef4050e3.ipynb.

This does NOT regenerate the 49 branches. It loads their saved outcomes and
performs one teacher-forced model pass to recover the true full-vocabulary
probabilities p1 and p2 at every baseline position.

Before running, make the previous ZIP available in either /kaggle/working or
/kaggle/input. The code finds it recursively.
"""

import csv
import gc
import json
import math
import shutil
import zipfile
from pathlib import Path
from typing import Any, Callable

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


MARGIN_TASK_ID = "HumanEval/26"
MARGIN_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
SOURCE_ARCHIVE_NAME = "exhaustive_branch_humaneval_26_qwen25_7b.zip"
MARGIN_OUTPUT_DIR = Path("/kaggle/working/margin_pilot_humaneval_26")
MARGIN_EXTRACT_DIR = Path("/kaggle/working/margin_pilot_source")
MARGIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MARGIN_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)


def find_source_artifacts() -> Path:
    """Use the prior output directory or safely extract its ZIP."""
    direct_dir = Path(
        "/kaggle/working/exhaustive_branch_humaneval_26_qwen25_7b"
    )
    required = {
        "baseline.json",
        "exhaustive_branches.jsonl",
        "exhaustive_summary.csv",
        "decision_report.json",
    }
    if direct_dir.is_dir() and required.issubset(
        {path.name for path in direct_dir.iterdir()}
    ):
        return direct_dir

    candidates = []
    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        if root.exists():
            candidates.extend(root.rglob(SOURCE_ARCHIVE_NAME))
    if not candidates:
        raise FileNotFoundError(
            f"Upload {SOURCE_ARCHIVE_NAME} to Kaggle Input or Working first."
        )

    archive_path = sorted(candidates, key=lambda path: len(str(path)))[0]
    print(f"Using source archive: {archive_path}")
    with zipfile.ZipFile(archive_path) as archive:
        names = {Path(name).name for name in archive.namelist()}
        if not required.issubset(names):
            raise RuntimeError(
                f"Source ZIP is missing: {sorted(required - names)}"
            )
        for info in archive.infolist():
            name = Path(info.filename)
            if name.is_absolute() or ".." in name.parts:
                raise RuntimeError(f"Unsafe ZIP path: {info.filename}")
        archive.extractall(MARGIN_EXTRACT_DIR)
    return MARGIN_EXTRACT_DIR


def read_jsonl_file(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def binary_ranking_metrics(
    rows: list[dict[str, Any]],
    score: Callable[[dict[str, Any]], float],
    descending: bool,
) -> dict[str, Any]:
    """Compute descriptive ranking metrics without sklearn."""
    ranked = sorted(rows, key=score, reverse=descending)
    labels = [int(row["top2_passed"]) for row in ranked]
    positives = sum(labels)
    negatives = len(labels) - positives
    if positives == 0 or negatives == 0:
        raise RuntimeError("Both positive and negative outcomes are required.")

    average_precision = sum(
        sum(labels[: index + 1]) / (index + 1)
        for index, label in enumerate(labels)
        if label
    ) / positives

    positive_scores = [
        score(row) for row in rows if row["top2_passed"]
    ]
    negative_scores = [
        score(row) for row in rows if not row["top2_passed"]
    ]
    pairwise_wins = 0.0
    for positive in positive_scores:
        for negative in negative_scores:
            if positive == negative:
                pairwise_wins += 0.5
            elif (positive > negative) == descending:
                pairwise_wins += 1.0
    auroc = pairwise_wins / (positives * negatives)

    ranks = {
        row["position"]: index
        for index, row in enumerate(ranked, start=1)
    }
    return {
        "auroc": auroc,
        "average_precision": average_precision,
        "positive_ranks": {
            str(row["position"]): ranks[row["position"]]
            for row in rows
            if row["top2_passed"]
        },
        "top_k": {
            str(k): {
                "positions": [row["position"] for row in ranked[:k]],
                "recoveries": sum(labels[:k]),
                "recall": sum(labels[:k]) / positives,
                "precision": sum(labels[:k]) / k,
            }
            for k in (1, 3, 5, 10, 15)
        },
    }


def ordinal_ranks(values: list[float], descending: bool) -> list[int]:
    """Deterministic ordinal ranks; ties keep position order."""
    order = sorted(
        range(len(values)),
        key=lambda index: values[index],
        reverse=descending,
    )
    ranks = [0] * len(values)
    for rank, index in enumerate(order):
        ranks[index] = rank
    return ranks


@torch.inference_mode()
def recompute_full_vocabulary_probabilities(
    model: Any,
    tokenizer: Any,
    prompt: str,
    baseline_token_ids: list[int],
) -> list[dict[str, Any]]:
    """Teacher-force the baseline and recover p1/p2 for every generated token."""
    input_device = model_input_device(model)
    prompt_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(input_device)
    baseline_ids = torch.tensor(
        [baseline_token_ids],
        dtype=torch.long,
        device=input_device,
    )
    full_ids = torch.cat([prompt_ids, baseline_ids], dim=1)
    prompt_length = prompt_ids.shape[1]

    outputs = model(full_ids, use_cache=False)
    # Index prompt_length - 1 predicts generated token 0.
    predicted_logits = outputs.logits[
        0,
        prompt_length - 1 : prompt_length - 1 + len(baseline_token_ids),
        :,
    ].float()
    log_probabilities = torch.log_softmax(predicted_logits, dim=-1)
    probabilities = log_probabilities.exp()
    top_probabilities, top_ids = torch.topk(probabilities, k=2, dim=-1)
    entropies = -(probabilities * log_probabilities).sum(dim=-1)
    top_logits = torch.gather(predicted_logits, 1, top_ids)

    results = []
    for position in range(len(baseline_token_ids)):
        p1 = float(top_probabilities[position, 0].item())
        p2 = float(top_probabilities[position, 1].item())
        z1 = float(top_logits[position, 0].item())
        z2 = float(top_logits[position, 1].item())
        results.append(
            {
                "position": position,
                "recomputed_top1_id": int(top_ids[position, 0].item()),
                "recomputed_top2_id": int(top_ids[position, 1].item()),
                "p1": p1,
                "p2": p2,
                "probability_margin": p1 - p2,
                "relative_probability_margin": (p1 - p2) / max(p1, 1e-30),
                "top2_to_top1_ratio": p2 / max(p1, 1e-30),
                "top2_mass": p1 + p2,
                "recomputed_logit_margin": z1 - z2,
                "recomputed_entropy": float(entropies[position].item()),
            }
        )

    del outputs, predicted_logits, log_probabilities, probabilities
    torch.cuda.empty_cache()
    return results


def make_scatter(rows: list[dict[str, Any]], output_path: Path) -> bool:
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib unavailable; skipping scatter plot.")
        return False

    colors = [
        "#198754" if row["top2_passed"] else "#9aa0a6"
        for row in rows
    ]
    plt.figure(figsize=(8, 5.5))
    plt.scatter(
        [row["probability_margin"] for row in rows],
        [row["recomputed_entropy"] for row in rows],
        c=colors,
        alpha=0.85,
    )
    for row in rows:
        if row["top2_passed"]:
            plt.annotate(
                str(row["position"]),
                (row["probability_margin"], row["recomputed_entropy"]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=9,
                fontweight="bold",
            )
    plt.xlabel("True probability margin p1 - p2 (smaller = more ambiguous)")
    plt.ylabel("Token entropy (larger = more uncertain)")
    plt.title("HumanEval/26: uncertainty signals and top-2 recoveries")
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(output_path, dpi=180)
    plt.close()
    return True


def run_margin_pilot() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle GPU accelerator.")

    source_dir = find_source_artifacts()
    baseline = json.loads(
        (source_dir / "baseline.json").read_text(encoding="utf-8")
    )
    branch_rows = read_jsonl_file(
        source_dir / "exhaustive_branches.jsonl"
    )
    if baseline["task_id"] != MARGIN_TASK_ID:
        raise RuntimeError(f"Unexpected task: {baseline['task_id']}")
    if len(branch_rows) != 49 or len(
        {row["position"] for row in branch_rows}
    ) != 49:
        raise RuntimeError("Expected 49 unique exhaustive branch records.")

    task = load_tasks(MARGIN_TASK_ID, num_tasks=1)[0]

    gc.collect()
    torch.cuda.empty_cache()
    print(f"Loading {MARGIN_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(MARGIN_MODEL)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MARGIN_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()
    prompt = build_prompt(task, tokenizer)

    recomputed = recompute_full_vocabulary_probabilities(
        model,
        tokenizer,
        prompt,
        baseline["token_ids"],
    )
    by_position = {row["position"]: row for row in recomputed}
    trace_by_position = {
        row["position"]: row for row in baseline["trace"]
    }

    joined = []
    mismatch_rows = []
    for branch in sorted(branch_rows, key=lambda row: row["position"]):
        position = branch["position"]
        probability_row = by_position[position]
        trace_row = trace_by_position[position]
        mismatch = {
            "position": position,
            "stored_top1": trace_row["top1_id"],
            "new_top1": probability_row["recomputed_top1_id"],
            "stored_top2": trace_row["top2_id"],
            "new_top2": probability_row["recomputed_top2_id"],
        }
        if (
            mismatch["stored_top1"] != mismatch["new_top1"]
            or mismatch["stored_top2"] != mismatch["new_top2"]
        ):
            mismatch_rows.append(mismatch)

        joined.append(
            {
                **branch,
                **probability_row,
                "stored_entropy": branch["entropy"],
                "stored_logit_margin": branch["margin"],
                "entropy_abs_error": abs(
                    branch["entropy"]
                    - probability_row["recomputed_entropy"]
                ),
                "logit_margin_abs_error": abs(
                    branch["margin"]
                    - probability_row["recomputed_logit_margin"]
                ),
            }
        )

    if mismatch_rows:
        print(
            f"WARNING: {len(mismatch_rows)} top-token mismatches after reload."
        )
        print(json.dumps(mismatch_rows[:10], indent=2))

    entropy_values = [row["recomputed_entropy"] for row in joined]
    probability_margins = [row["probability_margin"] for row in joined]
    entropy_ranks = ordinal_ranks(entropy_values, descending=True)
    margin_ranks = ordinal_ranks(probability_margins, descending=False)
    denominator = max(len(joined) - 1, 1)
    for index, row in enumerate(joined):
        row["entropy_rank_fraction"] = entropy_ranks[index] / denominator
        row["margin_rank_fraction"] = margin_ranks[index] / denominator
        # Fixed label-free combination: equal average of the two ranks.
        row["combined_rank_uncertainty"] = 1.0 - 0.5 * (
            row["entropy_rank_fraction"] + row["margin_rank_fraction"]
        )

    metrics = {
        "entropy_high": binary_ranking_metrics(
            joined,
            score=lambda row: row["recomputed_entropy"],
            descending=True,
        ),
        "probability_margin_low": binary_ranking_metrics(
            joined,
            score=lambda row: row["probability_margin"],
            descending=False,
        ),
        "top2_probability_high": binary_ranking_metrics(
            joined,
            score=lambda row: row["p2"],
            descending=True,
        ),
        "top2_to_top1_ratio_high": binary_ranking_metrics(
            joined,
            score=lambda row: row["top2_to_top1_ratio"],
            descending=True,
        ),
        "combined_rank_entropy_margin": binary_ranking_metrics(
            joined,
            score=lambda row: row["combined_rank_uncertainty"],
            descending=True,
        ),
    }

    correlation = float(
        np.corrcoef(entropy_values, probability_margins)[0, 1]
    )
    positive_details = [
        {
            "position": row["position"],
            "entropy": row["recomputed_entropy"],
            "p1": row["p1"],
            "p2": row["p2"],
            "probability_margin": row["probability_margin"],
            "top2_to_top1_ratio": row["top2_to_top1_ratio"],
            "top1_token": row["top1_token"],
            "top2_token": row["top2_token"],
        }
        for row in joined
        if row["top2_passed"]
    ]
    report = {
        "task_id": MARGIN_TASK_ID,
        "model": MARGIN_MODEL,
        "positions": len(joined),
        "recoveries": sum(row["top2_passed"] for row in joined),
        "top_token_mismatches_after_reload": len(mismatch_rows),
        "max_entropy_recompute_error": max(
            row["entropy_abs_error"] for row in joined
        ),
        "max_logit_margin_recompute_error": max(
            row["logit_margin_abs_error"] for row in joined
        ),
        "pearson_entropy_vs_probability_margin": correlation,
        "positive_details": positive_details,
        "ranking_metrics": metrics,
        "interpretation_rule": {
            "margin": "smaller p1-p2 means greater top-2 ambiguity",
            "entropy": "larger entropy means greater full-distribution uncertainty",
            "combined": "fixed equal-weight average of within-problem ranks; no labels used",
        },
    }

    report_path = MARGIN_OUTPUT_DIR / "margin_pilot_report.json"
    csv_path = MARGIN_OUTPUT_DIR / "margin_positions.csv"
    plot_path = MARGIN_OUTPUT_DIR / "margin_entropy_scatter.png"
    report_path.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    excluded_from_csv = {"top2_code", "error"}
    csv_fields = sorted(
        {
            key
            for row in joined
            for key in row
            if key not in excluded_from_csv
        }
    )
    with csv_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=csv_fields)
        writer.writeheader()
        for row in joined:
            writer.writerow({key: row.get(key) for key in csv_fields})

    plot_created = make_scatter(joined, plot_path)
    archive_path = shutil.make_archive(
        str(MARGIN_OUTPUT_DIR),
        "zip",
        root_dir=MARGIN_OUTPUT_DIR,
    )

    print("\n=== MARGIN PILOT REPORT ===")
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print("\nDownload and send back:")
    print(archive_path)
    if plot_created:
        print(f"Scatter plot: {plot_path}")


run_margin_pilot()


Loading Qwen/Qwen2.5-Coder-7B-Instruct...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[
  {
    "position": 28,
    "stored_top1": 1372,
    "new_top1": 1629,
    "stored_top2": 1629,
    "new_top2": 1372
  }
]

=== MARGIN PILOT REPORT ===
{
  "task_id": "HumanEval/26",
  "model": "Qwen/Qwen2.5-Coder-7B-Instruct",
  "positions": 49,
  "recoveries": 3,
  "top_token_mismatches_after_reload": 1,
  "max_entropy_recompute_error": 0.00312197208404541,
  "max_logit_margin_recompute_error": 0.03125,
  "pearson_entropy_vs_probability_margin": -0.9547017301820563,
  "positive_details": [
    {
      "position": 4,
      "entropy": 0.19993337988853455,
      "p1": 0.9598875045776367,
      "p2": 0.025580130517482758,
      "probability_margin": 0.934307374060154,
      "top2_to_top1_ratio": 0.02664909210245252,
      "top1_token": "\n\n\n",
      "top2_token": "\n"
    },
    {
      "position": 5,
      "entropy": 0.0022935837041586637,
      "p1": 0.9997833371162415,
      "p2": 0.00014201307203620672,
      "probability_margin": 0.9996413240442052,
      "top2_to_top1_ratio": 0